# Part 2 : La Régression Logistique

**Durée estimée : 2h30**

## 🎯 Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Expliquer** pourquoi la régression linéaire ne fonctionne pas pour la classification
2. **Décrire** comment la fonction sigmoid transforme un score en probabilité
3. **Construire** un modèle de régression logistique avec scikit-learn
4. **Ajuster** les hyperparamètres clés (C, penalty, solver)
5. **Évaluer** le modèle avec les métriques appropriées

---

## 🌍 Hook : Comment Gmail sait-il que ce message est un spam ?

Chaque jour, votre boîte email reçoit des dizaines de messages. Certains arrivent dans votre boîte de réception, d'autres sont automatiquement envoyés dans les **spams**.

**Question :** Comment Gmail (ou Outlook) peut-il décider en une fraction de seconde si un email est légitime ou spam ?

*(Prenez 30 secondes pour réfléchir...)*

<details>
<summary>🤔 Votre intuition ?</summary>

### 🔑 Réponse

Gmail analyse des **caractéristiques** de l'email :
- Contient-il des mots suspects ("gratuit", "gagnez", "urgent") ?
- L'expéditeur est-il dans vos contacts ?
- Y a-t-il beaucoup de liens ?
- Le format ressemble-t-il à un spam connu ?

Ensuite, un modèle de **classification** prédit : **Spam** ou **Non-spam**.

C'est exactement ce que fait la **régression logistique** : prédire une catégorie (et non un nombre comme la régression linéaire).

</details>

### La détection de spam : un cas d'étude réel

Selon une [étude IEEE 2024](https://ieeexplore.ieee.org/abstract/document/11013300/), la régression logistique avec TF-IDF peut identifier des mots-clés critiques comme **"pill"**, **"PHP"** et **"Businessweek"** qui indiquent un pattern de spam.

Une autre [étude 2024](https://link.springer.com/article/10.1007/s10207-023-00756-1) utilisant un ensemble de modèles avec régression logistique comme méta-classifieur a atteint **98.8% de précision** sur la détection de spam.

| Caractéristique de l'email | Indice de spam |
|---------------------------|----------------|
| Contient "FREE" en majuscules | Fort |
| Beaucoup de points d'exclamation | Fort |
| Expéditeur inconnu | Moyen |
| Lien vers un domaine suspect | Fort |
| Adresse dans les contacts | Faible |

---

## 2.1 Intuition : Pourquoi pas la régression linéaire ?

Dans la Part 1, nous avons utilisé la régression linéaire pour prédire un **prix** — un nombre continu.

Mais ici, notre problème est différent. On veut prédire : **Spam ou Non-spam**. C'est une **catégorie**, pas un nombre.

**Question :** Pourquoi ne pas simplement coder Non-spam = 0 et Spam = 1, puis utiliser la régression linéaire ?

Essayons :

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split

# Données simplifiées : score de "spam-itude" d'un email
np.random.seed(42)
score_spam = np.concatenate([
    np.random.uniform(0, 4, 25),    # Non-spams (scores bas)
    np.random.uniform(6, 10, 25)    # Spams (scores élevés)
])
est_spam = np.concatenate([np.zeros(25), np.ones(25)])  # 0 = non-spam, 1 = spam

# Essayer la régression linéaire
model_lineaire = LinearRegression()
model_lineaire.fit(score_spam.reshape(-1, 1), est_spam)

# Visualisation
x_line = np.linspace(-1, 11, 100)
y_pred_lineaire = model_lineaire.predict(x_line.reshape(-1, 1))

plt.figure(figsize=(10, 5))
plt.scatter(score_spam[est_spam==0], est_spam[est_spam==0], c='green', s=80, label='Non-spam (0)', alpha=0.7)
plt.scatter(score_spam[est_spam==1], est_spam[est_spam==1], c='red', s=80, label='Spam (1)', alpha=0.7)
plt.plot(x_line, y_pred_lineaire, 'b-', linewidth=2, label='Régression linéaire')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
plt.fill_between(x_line, -0.5, 0, alpha=0.1, color='purple', label='Zone < 0 (impossible !)')
plt.fill_between(x_line, 1, 1.5, alpha=0.1, color='orange', label='Zone > 1 (impossible !)')
plt.xlabel('Score "spam-itude"')
plt.ylabel('Classe (0=Non-spam, 1=Spam)')
plt.title('Problème : La régression linéaire peut prédire des valeurs < 0 ou > 1 !')
plt.legend(loc='center right')
plt.ylim(-0.5, 1.5)
plt.grid(True, alpha=0.3)
plt.show()

print(f"⚠️ Prédiction pour score = -1 : {model_lineaire.predict([[-1]])[0]:.2f}")
print(f"⚠️ Prédiction pour score = 12 : {model_lineaire.predict([[12]])[0]:.2f}")
print("\nCes valeurs n'ont pas de sens comme probabilités !")

### Le problème

La régression linéaire peut prédire **n'importe quelle valeur** : -0.15, 1.23, etc.

Or, une **probabilité** doit être entre 0 et 1 !

```
┌─────────────────────────────────────────────────────────────────────┐
│ POURQUOI LA RÉGRESSION LINÉAIRE NE MARCHE PAS POUR LA CLASSIFICATION│
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Régression linéaire : y = wx + b                                 │
│                                                                     │
│   ✗ Peut prédire -0.15  →  Probabilité négative ? IMPOSSIBLE       │
│   ✗ Peut prédire 1.23   →  Plus de 100% ? IMPOSSIBLE               │
│                                                                     │
│   BESOIN : Une fonction qui "écrase" tout vers [0, 1]              │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### La solution : La fonction Sigmoid

La fonction **sigmoid** transforme n'importe quel nombre en une valeur entre 0 et 1 :

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

| Si l'entrée est... | La sigmoid retourne... | Interprétation |
|--------------------|------------------------|----------------|
| Très négative (-10, -5...) | Proche de 0 | "Très probablement NON" |
| Autour de 0 | Environ 0.5 | "Je ne sais pas, 50/50" |
| Très positive (+5, +10...) | Proche de 1 | "Très probablement OUI" |

In [ ]:
# Visualiser la fonction sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)
s = sigmoid(z)

plt.figure(figsize=(10, 5))
plt.plot(z, s, 'b-', linewidth=3, label='Fonction Sigmoid')
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Seuil = 0.5')
plt.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
plt.axhline(y=1, color='gray', linestyle=':', alpha=0.5)
plt.scatter([0], [0.5], s=100, c='red', zorder=5)
plt.xlabel('Score z (sortie de la régression linéaire)')
plt.ylabel('Probabilité σ(z)')
plt.title('La fonction Sigmoid : transformer n\'importe quel nombre en probabilité [0, 1]')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

### Régression logistique = Régression linéaire + Sigmoid

```
┌─────────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : Régression Logistique                                   │
│                                                                         │
│ La **régression logistique** est un algorithme de **classification**    │
│ qui prédit la probabilité qu'une observation appartienne à une classe.  │
│                                                                         │
│ Elle combine :                                                          │
│ 1. Une régression linéaire : z = w₁x₁ + w₂x₂ + ... + b                 │
│ 2. Une fonction sigmoid : P(classe=1) = 1 / (1 + e^(-z))               │
│                                                                         │
│ Malgré son nom, c'est un algorithme de **classification**, pas de      │
│ régression !                                                            │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Comparer régression linéaire vs logistique
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax in axes:
    ax.scatter(score_spam[est_spam==0], est_spam[est_spam==0], c='green', s=80, label='Non-spam', alpha=0.7)
    ax.scatter(score_spam[est_spam==1], est_spam[est_spam==1], c='red', s=80, label='Spam', alpha=0.7)

# Régression linéaire
axes[0].plot(x_line, y_pred_lineaire, 'b-', linewidth=2)
axes[0].set_title('Régression Linéaire ❌\n(valeurs hors [0,1])')
axes[0].set_ylim(-0.3, 1.3)

# Régression logistique
model_log = LogisticRegression()
model_log.fit(score_spam.reshape(-1, 1), est_spam)
y_pred_log = model_log.predict_proba(x_line.reshape(-1, 1))[:, 1]
axes[1].plot(x_line, y_pred_log, 'orange', linewidth=2)
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Seuil 0.5')
axes[1].set_title('Régression Logistique ✅\n(probabilités entre 0 et 1)')
axes[1].set_ylim(-0.1, 1.1)

for ax in axes:
    ax.set_xlabel('Score "spam-itude"')
    ax.set_ylabel('Probabilité d\'être spam')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 2.2 Construction du modèle

Appliquons le pattern fit/predict que nous avons appris au Chapitre 2.

In [ ]:
# Préparer les données
df = pd.DataFrame({'score': score_spam, 'est_spam': est_spam.astype(int)})

X = df[['score']]
y = df['est_spam']

# Séparation train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_train)} exemples")
print(f"Test: {len(X_test)} exemples")

In [ ]:
# Créer et entraîner le modèle
model = LogisticRegression()
model.fit(X_train, y_train)

print("✅ Modèle entraîné !")
print(f"\nCe que le modèle a appris :")
print(f"  - Coefficient : {model.coef_[0][0]:.4f}")
print(f"  - Intercept : {model.intercept_[0]:.4f}")

In [ ]:
# Prédire : classes vs probabilités
y_pred = model.predict(X_test)           # Classes : 0 ou 1
y_proba = model.predict_proba(X_test)    # Probabilités : [P(0), P(1)]

print("Prédictions sur le test set :")
print("─" * 60)
for i in range(min(5, len(X_test))):
    score = X_test.iloc[i]['score']
    proba_spam = y_proba[i][1]
    pred = "🚫 SPAM" if y_pred[i] == 1 else "✉️ Non-spam"
    vrai = "🚫 SPAM" if y_test.iloc[i] == 1 else "✉️ Non-spam"
    print(f"Score {score:.1f} → P(spam)={proba_spam:.1%} → {pred} (Réel: {vrai})")

### Le seuil de décision

Par défaut, le seuil est **0.5** : si P(spam) ≥ 50%, on prédit "spam".

Mais ce seuil peut être ajusté selon le contexte :
- **Gmail personnel** → seuil élevé (0.7) : on préfère laisser passer quelques spams que de bloquer un email important
- **Banque anti-fraude** → seuil bas (0.3) : on préfère bloquer des transactions légitimes que de rater une fraude

In [ ]:
# Ajuster le seuil manuellement
def predire_avec_seuil(model, X, seuil=0.5):
    """Prédit avec un seuil personnalisé."""
    probas = model.predict_proba(X)[:, 1]
    return (probas >= seuil).astype(int)

# Comparer différents seuils
for seuil in [0.3, 0.5, 0.7]:
    y_pred_seuil = predire_avec_seuil(model, X_test, seuil)
    accuracy = (y_pred_seuil == y_test).mean()
    print(f"Seuil {seuil} → Accuracy: {accuracy:.1%}")

---

## 2.3 Hyperparamètres

La régression logistique a plusieurs hyperparamètres importants :

| Hyperparamètre | Description | Valeur par défaut |
|----------------|-------------|-------------------|
| **C** | Force de régularisation inverse (plus petit = plus de régularisation) | 1.0 |
| **penalty** | Type de régularisation ('l1', 'l2', 'elasticnet', None) | 'l2' |
| **solver** | Algorithme d'optimisation | 'lbfgs' |
| **max_iter** | Nombre max d'itérations | 100 |

### C : Le contrôle de la régularisation

**C** contrôle le compromis entre :
- **Petit C** (ex: 0.01) → Forte régularisation → Modèle plus simple, risque d'underfitting
- **Grand C** (ex: 100) → Faible régularisation → Modèle plus complexe, risque d'overfitting

```
┌─────────────────────────────────────────────────────────────────────┐
│                 L'HYPERPARAMÈTRE C                                  │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   C petit (0.01)                    C grand (100)                   │
│   ─────────────────                 ─────────────────               │
│   • Forte régularisation            • Faible régularisation         │
│   • Coefficients plus petits        • Coefficients plus grands      │
│   • Modèle plus simple              • Modèle plus complexe          │
│   • Risque: underfitting            • Risque: overfitting           │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Impact de C sur les coefficients
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

# Dataset plus réaliste
cancer = load_breast_cancer()
X_cancer = StandardScaler().fit_transform(cancer.data)
y_cancer = cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42
)

# Comparer différentes valeurs de C
print("Impact de C sur le modèle :")
print("─" * 60)

for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    model_c = LogisticRegression(C=C, max_iter=1000, random_state=42)
    model_c.fit(X_train_c, y_train_c)
    
    train_score = model_c.score(X_train_c, y_train_c)
    test_score = model_c.score(X_test_c, y_test_c)
    coef_norm = np.linalg.norm(model_c.coef_)  # "Taille" des coefficients
    
    print(f"C={C:6.2f} → Train: {train_score:.3f} | Test: {test_score:.3f} | ||coef||: {coef_norm:.1f}")

### penalty : Type de régularisation

| Penalty | Description | Quand l'utiliser |
|---------|-------------|------------------|
| **'l2'** (défaut) | Ridge - pénalise les gros coefficients | Cas général |
| **'l1'** | Lasso - peut mettre des coefficients à 0 | Sélection de features |
| **'elasticnet'** | Combinaison L1 + L2 | Beaucoup de features corrélées |
| **None** | Pas de régularisation | Données abondantes, peu de features |

In [ ]:
# L1 vs L2 : impact sur les coefficients
print("Comparaison L1 vs L2 :")
print("─" * 60)

# L2 (Ridge)
model_l2 = LogisticRegression(penalty='l2', C=0.1, max_iter=1000, random_state=42)
model_l2.fit(X_train_c, y_train_c)
coefs_l2 = model_l2.coef_[0]
zeros_l2 = np.sum(np.abs(coefs_l2) < 0.01)

# L1 (Lasso) - nécessite solver='saga'
model_l1 = LogisticRegression(penalty='l1', C=0.1, solver='saga', max_iter=1000, random_state=42)
model_l1.fit(X_train_c, y_train_c)
coefs_l1 = model_l1.coef_[0]
zeros_l1 = np.sum(np.abs(coefs_l1) < 0.01)

print(f"L2 (Ridge) : {zeros_l2} coefficients ≈ 0 sur {len(coefs_l2)}")
print(f"L1 (Lasso) : {zeros_l1} coefficients ≈ 0 sur {len(coefs_l1)}")
print(f"\n→ L1 effectue une sélection de features automatique !")

### solver : Algorithme d'optimisation

| Solver | Supporte | Recommandé pour |
|--------|----------|------------------|
| **'lbfgs'** | L2, None | Petit à moyen dataset (défaut) |
| **'liblinear'** | L1, L2 | Petit dataset |
| **'saga'** | L1, L2, elasticnet | Grand dataset, régularisation flexible |
| **'newton-cg'** | L2, None | Multiclasse |

En pratique, **'lbfgs'** (défaut) fonctionne bien dans la plupart des cas.

---

## 2.4 Évaluation du modèle

Nous avons appris les métriques d'évaluation au Chapitre 2 (Part 3). Appliquons-les ici.

**Rappel :** Pour la classification, les métriques clés sont :
- **Accuracy** : % de prédictions correctes (attention aux classes déséquilibrées !)
- **Precision** : Parmi mes prédictions positives, combien sont vraies ?
- **Recall** : Parmi les vrais positifs, combien ai-je trouvés ?
- **F1-Score** : Équilibre precision-recall
- **ROC-AUC** : Capacité de discrimination (tous seuils)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)

# Entraîner le modèle final
model_final = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model_final.fit(X_train_c, y_train_c)

# Prédictions
y_pred_final = model_final.predict(X_test_c)
y_proba_final = model_final.predict_proba(X_test_c)[:, 1]

In [ ]:
# Fonction utilitaire (définie au Ch2 Part3)
def evaluer_classification(y_vrai, y_pred, y_proba=None, nom_modele="Modèle"):
    """Affiche toutes les métriques de classification."""
    print(f"\n📊 Évaluation : {nom_modele}")
    print("=" * 50)
    print(f"Accuracy  : {accuracy_score(y_vrai, y_pred):.4f}")
    print(f"Precision : {precision_score(y_vrai, y_pred):.4f}")
    print(f"Recall    : {recall_score(y_vrai, y_pred):.4f}")
    print(f"F1-Score  : {f1_score(y_vrai, y_pred):.4f}")
    if y_proba is not None:
        print(f"ROC-AUC   : {roc_auc_score(y_vrai, y_proba):.4f}")

# Évaluer
evaluer_classification(y_test_c, y_pred_final, y_proba_final, "Régression Logistique")

In [ ]:
# Rapport complet
print("\n📊 Rapport de Classification")
print("=" * 55)
print(classification_report(y_test_c, y_pred_final, target_names=['Maligne', 'Bénigne']))

In [ ]:
# Visualisations
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matrice de confusion
ConfusionMatrixDisplay.from_predictions(
    y_test_c, y_pred_final, 
    display_labels=['Maligne', 'Bénigne'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Matrice de Confusion')

# Courbe ROC
RocCurveDisplay.from_predictions(y_test_c, y_proba_final, ax=axes[1])
axes[1].plot([0, 1], [0, 1], 'k--', label='Modèle aléatoire')
axes[1].set_title(f'Courbe ROC (AUC = {roc_auc_score(y_test_c, y_proba_final):.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

### Analyse des résultats

Dans le contexte du diagnostic du cancer :
- **Le Recall est crucial** : on ne veut pas manquer un cancer (Faux Négatif = danger !)
- **La Precision** est aussi importante : éviter de stresser inutilement les patients (Faux Positifs)

Notre modèle a un bon équilibre avec un F1-Score élevé.

---

## 2.5 Classification Multiclasse

La régression logistique peut aussi gérer **plus de 2 classes** (One-vs-Rest ou Softmax).

In [ ]:
from sklearn.datasets import load_iris

# Dataset avec 3 classes
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.3, stratify=y_iris, random_state=42
)

# Entraîner (sklearn gère automatiquement le multiclasse)
model_multi = LogisticRegression(max_iter=200, random_state=42)
model_multi.fit(X_train_i, y_train_i)

# Évaluer
y_pred_i = model_multi.predict(X_test_i)
print("📊 Classification Multiclasse (Iris - 3 classes)")
print("=" * 55)
print(classification_report(y_test_i, y_pred_i, target_names=iris.target_names))

In [ ]:
# Probabilités par classe
exemple = X_test_i[0:1]
probas = model_multi.predict_proba(exemple)[0]

print("\nProbabilités pour un exemple :")
for classe, proba in zip(iris.target_names, probas):
    indicateur = " ✓" if proba == max(probas) else ""
    print(f"  P({classe:12}) = {proba:.1%}{indicateur}")

---

## 📚 Récapitulatif

### Ce que vous avez appris :

1. **Intuition** : La régression linéaire ne convient pas pour la classification → on ajoute la sigmoid
2. **Construction** : `LogisticRegression().fit(X_train, y_train)` + `.predict()` et `.predict_proba()`
3. **Hyperparamètres** :
   - **C** : contrôle la régularisation (petit = plus de régularisation)
   - **penalty** : type de régularisation (l2 par défaut, l1 pour sélection de features)
   - **solver** : algorithme d'optimisation (lbfgs par défaut)
4. **Évaluation** : Appliquer les métriques du Ch2 (confusion matrix, precision, recall, F1, ROC-AUC)

### Code essentiel :

```python
from sklearn.linear_model import LogisticRegression

# Créer et entraîner
model = LogisticRegression(C=1.0, penalty='l2', max_iter=1000)
model.fit(X_train, y_train)

# Prédire
y_pred = model.predict(X_test)           # Classes
y_proba = model.predict_proba(X_test)    # Probabilités

# Seuil personnalisé
seuil = 0.3
y_pred_custom = (model.predict_proba(X_test)[:, 1] >= seuil).astype(int)

# Évaluer
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))
```

---

## 🤔 Réflexion Métacognitive

1. Pouvez-vous expliquer la différence entre régression linéaire et régression logistique à un non-technicien ?
2. Dans quel cas ajusteriez-vous le seuil de décision ? Donnez un exemple concret.
3. Quelle valeur de C choisiriez-vous si votre modèle overfitte ?

---

## ➡️ Prochaine partie

Dans la **Part 3 : Arbres de Décision & Random Forest**, nous allons découvrir une approche totalement différente. Les modèles linéaires fonctionnent bien quand la relation est... linéaire ! Mais que faire quand ce n'est pas le cas ?

Les **arbres de décision** offrent une approche très intuitive qui "pose des questions" sur les données.

---

**Sources :**
- [IEEE 2024 - Email Spam Detection Using Logistic Regression and Explainable AI](https://ieeexplore.ieee.org/abstract/document/11013300/)
- [Springer 2024 - Improving spam email classification using ensemble techniques](https://link.springer.com/article/10.1007/s10207-023-00756-1)
- [scikit-learn LogisticRegression Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)